In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from config import CAT_VARS, DATA, NUM_VARS
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
dataset_path = DATA / "processed" / "basic_data_prep"
datasets = {f.stem: pd.read_csv(f) for f in sorted(dataset_path.glob("*.csv"))}
for k, v in datasets.items():
    print(f"{k}: \n{v}\n")

In [ ]:
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), NUM_VARS),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_VARS),
    ]
)

model = Pipeline(
    [
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, C=np.inf)),
    ]
)

model.fit(datasets["X_train"], datasets["y_train"])

In [ ]:
# Comparison with bland inputs without any preprocessing.
# model = LogisticRegression(max_iter=1000, C=np.inf)
# model.fit(datasets["X_train"], datasets["y_train"])

# Validate results

In [ ]:
y_score_test = model.predict_proba(datasets["X_test"])[:, 1]
y_pred_test = model.predict(
    datasets["X_test"],
)
y_true_test = datasets["y_test"].to_numpy().ravel()
print(f"y_true_test: {y_true_test}")
print(f"y_pred_test: {y_pred_test}")
print(f"y_score_test: {y_score_test}")
y_score_train = model.predict_proba(datasets["X_train"])[:, 1]
y_true_train = datasets["y_train"].to_numpy().ravel()
y_pred_train = model.predict(
    datasets["X_train"],
)
print(f"y_true_train: {y_true_train}")
print(f"y_pred_train: {y_pred_train}")
print(f"y_score_train: {y_score_train}")

In [ ]:
auc_test = roc_auc_score(y_true_test, y_score_test)
gini_test = 2 * auc_test - 1
auc_train = roc_auc_score(y_true_train, y_score_train)
gini_train = 2 * auc_train - 1
conf_matrix_test = confusion_matrix(datasets["y_test"], y_pred_test).tolist()
conf_matrix_train = confusion_matrix(datasets["y_train"], y_pred_train).tolist()

metrics = {
    "test auc": auc_test, "test gini": gini_test, "test confusion matrix": conf_matrix_test,
    "train auc": auc_train, "train gini": gini_train, "train confusion matrix": conf_matrix_train,
}

print(metrics)

In [ ]:
fpr_test, tpr_test, thr_test = roc_curve(y_true_test, y_score_test)
fpr_train, tpr_train, thr_train = roc_curve(y_true_train, y_score_train)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        hoverinfo="skip",
        line={"color": "#B0B4BA", "width": 1, "dash": "dash"},
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr_test,
        y=tpr_test,
        mode="lines",
        customdata=thr_test,
        line={"color": "#2B6CB0", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr_train,
        y=tpr_train,
        mode="lines",
        customdata=thr_train,
        line={"color": "#FF0000", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title=f"ROC — Logistic: test (AUC {auc_test:.3f}, Gini {2 * auc_test - 1:.3f}),<br> train (AUC {auc_train:.3f}, Gini {2 * auc_train - 1:.3f})",
    xaxis={
        "title": "False positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
    },
    yaxis={
        "title": "True positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
        "scaleanchor": "x",
        "scaleratio": 1,
    },
    template="plotly_white",
    width=560,
    height=560,
    margin={"l": 60, "r": 30, "t": 60, "b": 60},
)
fig.show()

**Conclusion:** OK, so Gini is working well for the lowest scores but stops working quite as well around the middle range. Can try to get a better performing model to tell whether this is a limitation of the dataset or the model.